# ⚓ Anclaje humano de la métrica

Todo el laboratorio descansa sobre una nota automática: `relevancia = α·léxico + (1−α)·semántico`. Los bancos comparan con ella, la estadística decide con ella, la Fase B corona con ella. Pero **¿la nota coincide con lo que TÚ considerarías relevante?** Ese es el eslabón que nunca se ha verificado — este notebook lo verifica.

## Qué hace

1. Reconstruye un pipeline concreto (splitter + embedding + retriever) y recupera fragmentos para una muestra del **examen congelado** del documento.
2. Te los presenta **a ciegas** (sin la nota automática, para no sesgarte) y tú puntúas cada fragmento: ¿responde a la pregunta?
3. Compara tus etiquetas con la nota automática: correlación, el mejor `umbral_relevancia` según TU criterio, y el acuerdo del MRR humano vs el MRR de la máquina.
4. Persiste tus etiquetas en la memoria (colección `anclaje_humano`), así la sesión es **reanudable** y el gold set crece entre sesiones.

## Cómo interpretarlo

- **Correlación (Spearman) alta (≥0.6)** → la nota ordena como tú: los veredictos de los bancos son creíbles.
- **Umbral óptimo lejos de tu `umbral_relevancia`** → el MRR está contando aciertos que tú no aceptarías (o descartando los que sí): exporta `RAG_UMBRAL_RELEVANCIA` con el valor sugerido.
- **Desacuerdo sistemático en paráfrasis** (léxico bajo, tú dices "relevante") → sube el peso semántico (baja `RAG_PESO_EXACTITUD`).

Nota metodológica: tú juzgas **pregunta ↔ fragmento**; la máquina juzga **fragmento ↔ cita de la pregunta**. Esa diferencia es deliberada — es exactamente lo que se quiere validar: que anclar la nota a la cita sea un buen proxy del juicio humano sobre la pregunta.

## 0 · Preparación
Requiere el arranque A1–A7 de `entrenar.ipynb` ya ejecutado (o entorno local equivalente). Configura documento y stack a auditar — lo natural es auditar el **stack campeón** de tu última corrida.

In [ ]:
from pathlib import Path
import numpy as np

from rag_lab import CONFIG
from rag_lab.memory import Memoria
from rag_lab.corpus import preparar_sesion, hash_pdf
from rag_lab.fabricas import construir_embedding
from rag_lab.recuperacion import construir_recuperador
from rag_lab.bancos.embeddings_banco import _construir_splitter, EMBEDDING_REGISTRY
from rag_lab.evaluacion import obtener_embedding_juez, obtener_base_semantica
from rag_lab.metricas import (PESO_EXACTITUD, UMBRAL_RELEVANCIA, gramas_de,
                              cobertura, vector_normalizado, recalibrar_similitud)
from langchain_community.vectorstores import FAISS

# --- AJUSTA ESTO: documento y stack a auditar (usa el campeón de tu informe) ---
PDF = "pdfs/test_10.pdf"
STACK = {
    "splitter":  {"name": "recursive", "chunk_size": 1000, "chunk_overlap": 200},
    "embedding": EMBEDDING_REGISTRY["bge-m3"],
    "retriever": {"nombre": "ensemble", "k": 5, "params": {"w_bm25": 0.3}},
}
N_PREGUNTAS = 8      # preguntas de la muestra a etiquetar (sube para más señal)
SEMILLA = 20260705   # muestreo reproducible

mem = Memoria()
doc_hash = hash_pdf(PDF) or ""
etiqueta_stack = (f"{STACK['splitter']['name']}{STACK['splitter']['chunk_size']}"
                  f"|{STACK['embedding'].get('model','?').split('/')[-1]}"
                  f"|{STACK['retriever']['nombre']}k{STACK['retriever']['k']}")
print("Documento:", PDF, "| hash:", doc_hash[:12])
print("Stack auditado:", etiqueta_stack)

## 1 · Examen congelado + pipeline
Se reutiliza el examen congelado (mismas preguntas contra las que decidieron los bancos: el anclaje audita **lo mismo que se midió**). Si el documento aún no tiene examen, genera uno primero corriendo el gimnasio.

In [ ]:
congeladas = mem.preguntas_congeladas(doc_hash)
assert congeladas, ("Este documento no tiene examen congelado: corre primero una "
                    "sesión del gimnasio para generarlo y verificarlo.")
sesion = preparar_sesion(PDF, preguntas_predefinidas=congeladas)
print(f"{len(sesion.preguntas_test)} preguntas congeladas cargadas.")

splitter = _construir_splitter(STACK["splitter"])
chunks = splitter.split_documents(sesion.documentos)
modelo = construir_embedding(STACK["embedding"])
vectorstore = FAISS.from_documents(chunks, modelo)
recuperador = construir_recuperador(STACK["retriever"]["nombre"],
                                    STACK["retriever"]["params"],
                                    vectorstore, chunks, STACK["retriever"]["k"])
print(f"Pipeline listo: {len(chunks)} chunks.")

## 2 · Muestra + nota automática por fragmento
Para cada pregunta de la muestra se recuperan los k fragmentos y se calcula su relevancia automática con **las mismas funciones y la misma calibración** que usan los bancos (α, base semántica calibrada del árbitro). La nota queda guardada pero **no se te enseña** hasta después de etiquetar.

In [ ]:
rng = np.random.default_rng(SEMILLA)
idx = sorted(rng.choice(len(sesion.preguntas_test),
                        min(N_PREGUNTAS, len(sesion.preguntas_test)),
                        replace=False).tolist())

juez = obtener_embedding_juez()
base = obtener_base_semantica(juez)
cache = {}
alpha = float(PESO_EXACTITUD)
print(f"α={alpha} | base semántica calibrada={base} | umbral vigente={UMBRAL_RELEVANCIA}")

pares = []   # un registro por (pregunta, fragmento)
for qi in idx:
    q = sesion.preguntas_test[qi]
    cita = q.get("source_context", "")
    grams_cita = gramas_de(cita)
    emb_cita = vector_normalizado(cita, juez, cache)
    docs = recuperador.invoke(q.get("question", ""))
    for fi, d in enumerate(docs):
        texto = d.page_content or ""
        lex = cobertura(gramas_de(texto), grams_cita)
        v = vector_normalizado(texto, juez, cache)
        sem = recalibrar_similitud(float(v @ emb_cita), base)
        pares.append({"qi": qi, "fi": fi, "question": q.get("question", ""),
                      "texto": texto, "lex": round(lex, 4), "sem": round(sem, 4),
                      "rel_auto": round(alpha * lex + (1 - alpha) * sem, 4)})
print(f"{len(pares)} pares (pregunta, fragmento) preparados para etiquetar.")

## 3 · Sesión de etiquetado (a ciegas)
Escala: **0** = irrelevante · **1** = relacionado pero no responde · **2** = responde a la pregunta. También: **s** = saltar, **q** = parar (el progreso queda guardado; re-ejecuta la celda para continuar donde lo dejaste).

In [ ]:
import hashlib

def _clave(par):
    th = hashlib.md5(par["texto"].encode()).hexdigest()[:10]
    return f"{doc_hash[:10]}|{etiqueta_stack}|q{par['qi']}|f{par['fi']}|{th}"

ya = {e["clave"]: e for e in mem._leer().get("anclaje_humano", [])
      if e.get("doc_hash") == doc_hash and e.get("stack") == etiqueta_stack}
print(f"Etiquetas previas de este stack/documento: {len(ya)}")

pendientes = [p for p in pares if _clave(p) not in ya]
print(f"Pendientes de etiquetar: {len(pendientes)}\n")

for n, par in enumerate(pendientes, 1):
    print("=" * 70)
    print(f"[{n}/{len(pendientes)}] PREGUNTA: {par['question']}")
    print("-" * 70)
    print("FRAGMENTO:")
    print(par["texto"][:900] + ("..." if len(par["texto"]) > 900 else ""))
    r = input("¿Responde? 0=no 1=parcial 2=sí | s=saltar q=parar > ").strip().lower()
    if r == "q":
        print("Sesión pausada: el progreso está guardado.")
        break
    if r not in ("0", "1", "2"):
        continue
    mem._anadir("anclaje_humano", {
        "clave": _clave(par), "doc_hash": doc_hash, "stack": etiqueta_stack,
        "qi": par["qi"], "fi": par["fi"], "humano": int(r),
        "lex": par["lex"], "sem": par["sem"], "rel_auto": par["rel_auto"]})
else:
    print("\n¡Muestra completa etiquetada!")

## 4 · Análisis del anclaje
Se cruzan tus etiquetas con la nota automática. Todo se calcula con numpy puro (sin scipy): Spearman por rangos, barrido de umbral contra tu criterio, y comparación de MRR humano vs MRR de la máquina.

In [ ]:
ETIQUETAS = [e for e in mem._leer().get("anclaje_humano", [])
             if e.get("doc_hash") == doc_hash and e.get("stack") == etiqueta_stack]
assert len(ETIQUETAS) >= 10, f"Solo {len(ETIQUETAS)} etiquetas: etiqueta más pares antes de analizar."

h = np.array([e["humano"] for e in ETIQUETAS], dtype=float)
a = np.array([e["rel_auto"] for e in ETIQUETAS], dtype=float)
lex = np.array([e["lex"] for e in ETIQUETAS], dtype=float)
sem = np.array([e["sem"] for e in ETIQUETAS], dtype=float)

def spearman(x, y):
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    if rx.std() == 0 or ry.std() == 0:
        return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])

print(f"n = {len(h)} pares etiquetados")
print(f"Spearman(humano, nota combinada) = {spearman(h, a):.3f}")
print(f"Spearman(humano, solo léxico)    = {spearman(h, lex):.3f}")
print(f"Spearman(humano, solo semántico) = {spearman(h, sem):.3f}")
print("\n(Si el semántico correlaciona mucho mejor que el léxico con tu juicio,")
print(" hay margen bajando RAG_PESO_EXACTITUD; y viceversa.)")

In [ ]:
# Barrido de umbral: ¿qué umbral de la nota reproduce mejor TU 'relevante'?
# 'Relevante' humano = etiqueta >= 1 (cambia a >= 2 para el criterio estricto).
CRITERIO = 1
rel_h = h >= CRITERIO

mejor = None
print(f"{'umbral':>7} {'precision':>10} {'recall':>8} {'F1':>7}")
for u in np.arange(0.05, 0.96, 0.05):
    pred = a >= u
    tp = float(np.sum(pred & rel_h)); fp = float(np.sum(pred & ~rel_h))
    fn = float(np.sum(~pred & rel_h))
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    marca = ""
    if mejor is None or f1 > mejor[1]:
        mejor = (u, f1)
    if abs(u - UMBRAL_RELEVANCIA) < 0.026:
        marca = "   <- umbral vigente"
    print(f"{u:7.2f} {prec:10.2f} {rec:8.2f} {f1:7.2f}{marca}")

print(f"\nUmbral óptimo según tu criterio (F1): {mejor[0]:.2f} (F1={mejor[1]:.2f})")
if abs(mejor[0] - UMBRAL_RELEVANCIA) >= 0.10:
    print(f"=> Difiere del vigente ({UMBRAL_RELEVANCIA}). Sugerencia: "
          f"os.environ['RAG_UMBRAL_RELEVANCIA'] = '{mejor[0]:.2f}' antes de importar rag_lab.")
else:
    print(f"=> Compatible con el vigente ({UMBRAL_RELEVANCIA}): el MRR cuenta aciertos como tú.")

In [ ]:
# MRR humano vs MRR de la máquina, pregunta a pregunta
from collections import defaultdict
por_q = defaultdict(list)
for e in ETIQUETAS:
    por_q[e["qi"]].append(e)

filas, difs = [], []
for qi, lst in sorted(por_q.items()):
    lst = sorted(lst, key=lambda e: e["fi"])
    mrr_h = next((1.0 / (e["fi"] + 1) for e in lst if e["humano"] >= CRITERIO), 0.0)
    mrr_a = next((1.0 / (e["fi"] + 1) for e in lst if e["rel_auto"] >= UMBRAL_RELEVANCIA), 0.0)
    difs.append(mrr_a - mrr_h)
    filas.append((qi, mrr_h, mrr_a))

print(f"{'pregunta':>9} {'MRR humano':>11} {'MRR máquina':>12}")
for qi, mh, ma in filas:
    print(f"{qi:>9} {mh:>11.3f} {ma:>12.3f}{'   <- desacuerdo' if abs(ma-mh)>0.2 else ''}")
print(f"\nMedia humano={np.mean([f[1] for f in filas]):.3f} | "
      f"máquina={np.mean([f[2] for f in filas]):.3f} | "
      f"sesgo medio (máquina-humano)={np.mean(difs):+.3f}")
print("Sesgo positivo => la máquina se apunta aciertos que tú no le darías (umbral blando);")
print("negativo => descarta aciertos reales (umbral duro o α mal repartido).")

## 5 · Veredicto del anclaje

Con esto tienes el eslabón cerrado: **humano ⚓ nota automática ⚓ veredictos de los bancos**.

- Si el Spearman de la nota combinada supera al de sus dos partes por separado, la mezcla α está aportando — el diseño híbrido se justifica con tus propios datos.
- Las etiquetas quedan en `rag_memory` (colección `anclaje_humano`), acumulables entre sesiones y stacks: etiquetar 2-3 stacks distintos sobre 2-3 documentos (~100 pares, 20 minutos) da un gold set pequeño pero tuyo.
- Repite el anclaje cuando cambies algo estructural de la métrica (α, base, umbral, n-grama): es tu prueba de regresión contra el único juez que no se puede calibrar — tú.

**Importante**: si ajustas `RAG_UMBRAL_RELEVANCIA` o `RAG_PESO_EXACTITUD` a raíz de esto, los MRR nuevos cambian de escala respecto a los históricos — mismo aviso de siempre: limpia o reetiqueta la memoria antes de la siguiente tanda seria.